In [1]:
import pandas as pd
import numpy as np

In [2]:
ENTREPOT_PATH = '/home/tbadie/Bureau/data/data_entrepot_outils/'
donnees = {}

def import_dfs(df_names, path_data, sep = ','):
    i = 0
    for df_name in df_names: 
        donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)

tables = [
    'noeuds_realise',
    # 'entite_unique_par_sdc_nettoyage', 
    'zone',
    'parcelle'
    ]

# import des données
import_dfs(tables, ENTREPOT_PATH, sep = ',')

In [3]:
nd = donnees['noeuds_realise'][['id','culture_id','zone_id']].rename(columns={'id':'noeuds_realise_id'})
zone = donnees['zone'][['id','surface','parcelle_id']].rename(columns={'id':'zone_id'})
parcelle = donnees['parcelle'][['id','surface','sdc_id']].rename(columns={'id':'parcelle_id', 'surface':'surface_parcelle'})

In [4]:
df = nd.merge(zone, on = 'zone_id', how = 'left').merge(parcelle, on = 'parcelle_id', how = 'left')
df = df.loc[(df['sdc_id'].notna()) & (df['surface'] != 0)]

In [5]:
df['nb_itk_mm_zone'] = df.groupby("zone_id")["noeuds_realise_id"].transform("count")
df['surface_ponderee_zone'] = df['surface'] / df['nb_itk_mm_zone']

df['surface_ponderee_totale'] = df.groupby("sdc_id")["surface_ponderee_zone"].transform("sum")
df["surface_developpee_totale"] = df.groupby("sdc_id")["surface"].transform("sum")

df['poids_surface_ponderee'] = df['surface_ponderee_zone'] / df['surface_ponderee_totale']
df['poids_surface_developpee_agreg'] = df['surface'] / df['surface_ponderee_totale']
df['poids_surface_developpee_normalisee'] = df['surface'] / df['surface_developpee_totale']

df = df[['noeuds_realise_id', 'culture_id', 'sdc_id', 'poids_surface_ponderee', 'poids_surface_developpee_agreg', 'poids_surface_developpee_normalisee']]

In [6]:
df.loc[df['poids_surface_developpee_normalisee'] < df['poids_surface_ponderee']-0.01].sample(5)

,noeuds_realise_id,culture_id,sdc_id,poids_surface_ponderee,poids_surface_developpee_agreg,poids_surface_developpee_normalisee
44388,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_0f...,0.168990,0.168990,0.137803
94899,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_ec...,0.095898,0.095898,0.070358
132215,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_7f...,0.050051,0.050051,0.036862
23314,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_87...,0.029884,0.029884,0.016180
119403,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_a6...,0.047519,0.047519,0.034532


In [7]:
df.loc[df['poids_surface_developpee_normalisee'] > df['poids_surface_ponderee']+0.01].sample(5)

,noeuds_realise_id,culture_id,sdc_id,poids_surface_ponderee,poids_surface_developpee_agreg,poids_surface_developpee_normalisee
70082,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_9a...,0.043706,0.087413,0.074405
115270,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_34...,0.019650,0.058951,0.045764
112465,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_bd...,0.025337,0.050674,0.042828
132321,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_3d...,0.019682,0.039364,0.036575
56501,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_f3...,0.036048,0.072097,0.054672


In [8]:
df.groupby("sdc_id")["poids_surface_developpee_agreg"].apply("sum").sort_values(ascending=False)

sdc_id
fr.inra.agrosyst.api.entities.GrowingSystem_f3c49758-0a8d-484a-ba10-f8e353af3462    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_159ebcce-5b83-4814-8aab-6dab6c24b414    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_288c3360-04b0-4b39-9bd5-06a45a705110    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_e91191aa-93dc-476e-8821-2fcfeb42c545    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_44277df3-7353-4b81-b993-370a3b1337d9    6.0
                                                                                   ... 
fr.inra.agrosyst.api.entities.GrowingSystem_91c5f1d9-d9ac-442c-b3be-9f02de92a53f    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_93fb16d1-a082-4e18-8df4-2acb48c4f289    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_be7bb970-6569-4058-81db-c9d3b7f46cf8    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_d2e149dd-72f5-43f7-94b2-7ce6655034c0    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_d2615951-0cc8-4210-91e3-c36764d12848    1.0
Name: poids_surface_devel

In [9]:
list_sdc = ['fr.inra.agrosyst.api.entities.GrowingSystem_240d79e7-05d5-483b-b051-aab89f068ce8',
'fr.inra.agrosyst.api.entities.GrowingSystem_09a59c8b-c3a2-4e08-9a2d-b8a68eea77c2',
'fr.inra.agrosyst.api.entities.GrowingSystem_4b47dcf4-1418-49d5-8c6f-f2c1b4afb9df',
'fr.inra.agrosyst.api.entities.GrowingSystem_dbfd4211-8d4b-4021-8b02-5a09c06aad43',
'fr.inra.agrosyst.api.entities.GrowingSystem_0ce4d7e5-fe26-46c0-b9bd-4fa4f865fb37',
'fr.inra.agrosyst.api.entities.GrowingSystem_7f9cac9c-7ddd-44bf-bead-fcc78cde4e48',
'fr.inra.agrosyst.api.entities.GrowingSystem_21c3bd29-7480-4255-9399-f4620688fce2']

In [10]:
TEST_PATH = '/home/tbadie/Bureau/catalogue_script_agrosyst/02_outils/tests/data/test_get_poids_noeuds_realise/'

parcelle2 = donnees['parcelle'].loc[donnees['parcelle']['sdc_id'].isin(list_sdc)]

zone2 = donnees['zone'].loc[donnees['zone']['parcelle_id'].isin(parcelle2['id'])]

nd2 = donnees['noeuds_realise'].loc[donnees['noeuds_realise']['zone_id'].isin(zone2['id'])]

parcelle2.to_csv(TEST_PATH + 'parcelle.csv', index=False, sep=',')
zone2.to_csv(TEST_PATH + 'zone.csv', index=False, sep=',')
nd2.to_csv(TEST_PATH + 'noeuds_realise.csv', index=False, sep=',')

In [11]:
del donnees
donnees = {}
import_dfs(tables, TEST_PATH, sep = ',')

In [12]:
nd = donnees['noeuds_realise'][['id','culture_id','zone_id']].rename(columns={'id':'noeuds_realise_id'})
zone = donnees['zone'][['id','surface','parcelle_id']].rename(columns={'id':'zone_id'})
parcelle = donnees['parcelle'][['id','surface','sdc_id']].rename(columns={'id':'parcelle_id', 'surface':'surface_parcelle'})

df = nd.merge(zone, on = 'zone_id', how = 'left').merge(parcelle, on = 'parcelle_id', how = 'left')
df = df.loc[(df['sdc_id'].notna()) & (df['surface'] != 0)]

df['nb_itk_mm_zone'] = df.groupby("zone_id")["noeuds_realise_id"].transform("count")
df['surface_ponderee_zone'] = df['surface'] / df['nb_itk_mm_zone']

df['surface_ponderee_totale'] = df.groupby("sdc_id")["surface_ponderee_zone"].transform("sum")
df["surface_developpee_totale"] = df.groupby("sdc_id")["surface"].transform("sum")

df['poids_surface_ponderee'] = df['surface_ponderee_zone'] / df['surface_ponderee_totale']
df['poids_surface_developpee_agreg'] = df['surface'] / df['surface_ponderee_totale']
df['poids_surface_developpee_normalisee'] = df['surface'] / df['surface_developpee_totale']

df = df[['noeuds_realise_id', 'culture_id', 'sdc_id', 'poids_surface_ponderee', 'poids_surface_developpee_agreg', 'poids_surface_developpee_normalisee']]

df

,noeuds_realise_id,culture_id,sdc_id,poids_surface_ponderee,poids_surface_developpee_agreg,poids_surface_developpee_normalisee
0,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_24...,0.003723,0.014891,0.006920
1,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_24...,0.005406,0.016218,0.007537
2,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_4b...,0.030488,0.060976,0.043251
3,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_4b...,0.037548,0.075096,0.053267
4,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_4b...,0.035302,0.070603,0.050080
...,...,...,...,...,...,...
116,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_7f...,0.020056,0.040113,0.023774
117,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_7f...,0.056497,0.056497,0.033484
118,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_7f...,0.095763,0.095763,0.056755
119,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_7f...,0.134322,0.268644,0.159216


In [13]:
def creer_df_tests(df, test_id, nb_par_colonne):
    lignes = []

    for colonne, n in nb_par_colonne.items():

        if colonne not in df.columns:
            raise ValueError(f"La colonne '{colonne}' n'existe pas.")

        serie = df[colonne].dropna()

        if n > len(serie):
            raise ValueError(
                f"Impossible de tirer {n} valeurs sans remise dans la colonne '{colonne}' "
                f"(seulement {len(serie)} valeurs disponibles)."
            )

        echantillon = serie.sample(n=n, replace=False)

        for idx, valeur in echantillon.items():
            lignes.append({
                "test_id": test_id,
                "index": idx,
                "valeur": valeur,
                "resultat": None,   # colonne vide
                "colonne": colonne
            })

    return pd.DataFrame(lignes)

In [18]:
nb_par_colonne = {
            "culture_id": 3,
            "sdc_id": 3,
            "poids_surface_ponderee": 25,
            "poids_surface_developpee_agreg": 25,
            "poids_surface_developpee_normalisee": 25
        }

final_TU = creer_df_tests(df, 'test_get_poids_noeuds_realise', nb_par_colonne)

final_TU.to_csv('/home/tbadie/Bureau/TU_a_utiliser.csv', index=False)